# MLflow Example: Tracking Two Model Training Runs

In [64]:
# %pip installs into the notebook's own Python (the kernel)
%pip install scikit-learn==1.5.1

Note: you may need to restart the kernel to use updated packages.


In [65]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [66]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [67]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [68]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [69]:
%pip install mlflow==3.18.0

  Using cached graphql_core-3.2.14-py3-none-any.whl.metadata (11 kB)
Using cached graphql_core-3.2.14-py3-none-any.whl (279 kB)
  Attempting uninstall: graphql-core
    Found existing installation: graphql-core 3.3.2
    Uninstalling graphql-core-3.3.2:
      Successfully uninstalled graphql-core-3.3.2
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gql 4.4.0 requires graphql-core<3.4,>=3.3.0a3, but you have graphql-core 3.2.14 which is incompatible.


In [70]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [71]:
%pip install dagshub

  Using cached graphql_core-3.3.2-py3-none-any.whl.metadata (11 kB)
Using cached graphql_core-3.3.2-py3-none-any.whl (366 kB)
  Attempting uninstall: graphql-core
    Found existing installation: graphql-core 3.2.14
    Uninstalling graphql-core-3.2.14:
      Successfully uninstalled graphql-core-3.2.14
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphene 3.4.3 requires graphql-core<3.3,>=3.1, but you have graphql-core 3.3.2 which is incompatible.
graphql-relay 3.2.0 requires graphql-core<3.3,>=3.2, but you have graphql-core 3.3.2 which is incompatible.


In [72]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='campoceblo', repo_name='MLFlow-lab', mlflow=True)

Initialized MLflow to track repo "campoceblo/MLFlow-lab"

Repository campoceblo/MLFlow-lab initialized!

In [73]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [74]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/4fecf6a2e9064b93ae5667968bb8ea4b', creation_time=1794840846931, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794840846931, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [75]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name): # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(
            model,
            name=run_name.replace(".", "_"),  # model names cannot contain dots
            skops_trusted_types=["sklearn.tree._tree.Tree"],
            registered_model_name="digits-classifier",  # used by the Docker build
        )

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [76]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(
	n_estimators=20,
	max_depth=5,
	run_name="random_forest_model_run_1"
)

Training and Logging Model 1...


Successfully registered model 'digits-classifier'.
2026/11/16 16:26:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-classifier, version 1
Created version '1' of model 'digits-classifier'.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/campoceblo/MLFlow-lab.mlflow/#/experiments/0/runs/5f5beb8e7ecd49dbac691c449ea8b2a6
🧪 View experiment at: https://dagshub.com/campoceblo/MLFlow-lab.mlflow/#/experiments/0


In [77]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")


Training and Logging Model 2...


Registered model 'digits-classifier' already exists. Creating a new version of this model...
2026/11/16 16:26:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-classifier, version 2
Created version '2' of model 'digits-classifier'.


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/campoceblo/MLFlow-lab.mlflow/#/experiments/0/runs/37eb6480276c484db40e40163000fb04
🧪 View experiment at: https://dagshub.com/campoceblo/MLFlow-lab.mlflow/#/experiments/0


In [78]:
# View the results on DagsHub
print("Runs:  https://dagshub.com/campoceblo/MLFlow-lab/experiments")
print("Registered model 'digits-classifier': DagsHub repo > Experiments > Models")

Runs:  https://dagshub.com/campoceblo/MLFlow-lab/experiments
Registered model 'digits-classifier': DagsHub repo > Experiments > Models
